# PLANTILLA CALIDAD DE DATOS

## IMPORTAR PAQUETES

In [ ]:
import numpy as np
import pandas as pd
from janitor import clean_names

#Automcompletar rapido
%config IPCompleter.greedy=True

## IMPORTAR LOS DATOS

In [ ]:
ruta_proyecto = 'C:/Users/vifra/vidal cientifico de datos/Portafolio ML/Proyecto leadscoring/00_PROYECTO_LEADSCORING/'

In [ ]:
nombre_fichero_datos = 'trabajo.csv'

In [ ]:
ruta_completa = ruta_proyecto + '/02_Datos/03_Trabajo/' + nombre_fichero_datos

df = pd.read_csv(ruta_completa,index_col=0)
df

## VISION GENERAL

In [ ]:
df.info()

## TIPOS DE DATOS

### Identificacion

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.dtypes

### Correccion

In [ ]:
df = df.astype({'visitas_total':'Int64'})

df.dtypes

## VALORES UNICOS

### Identificacion

In [ ]:
df.conociste_revista.value_counts()

In [ ]:
df.nunique().sort_values()

### Correccion

In [ ]:
df.drop(columns = ['conociste_youtube','conociste_revista','conociste_periodico'], inplace = True)
df.nunique().sort_values()

## DUPLICADOS

### Identificacion

In [ ]:
df.duplicated().sum()

### Correccion

In [ ]:
df.drop_duplicates(inplace = True)

## SEPARAR NUMERICAS Y CATEGORICAS

### Categoricas

In [ ]:
cat = df.select_dtypes(exclude = 'number').copy()

### Numericas

In [ ]:
num = df.select_dtypes(include='number').copy()

In [ ]:
num

## GESTION DE CATEGORICAS

### Nulos

#### Identificacion

In [ ]:
cat.isna().sum().sort_values(ascending = False)

#### Correccion

In [ ]:
cat.ocupacion.value_counts()

##### Imputar por un valor

In [ ]:
cat.columns.to_list()

###### Variables a imputar

In [ ]:
var_imputar_valor = ['origen',
                     'fuente',
                     'no_enviar_email',
                     'no_llamar',
                     'ult_actividad',
                     'conociste_google',
                     'conociste_facebook',
                     'conociste_referencias',
                     'descarga_lm']

###### Valor por el que imputar

In [ ]:
valor = 'Desconocido'

###### Imputar

In [ ]:
 cat[var_imputar_valor] = cat[var_imputar_valor].fillna(valor)

##### Imputar por la moda

###### Variables a imputar

In [ ]:
var_imputar_moda = ['ocupacion','ambito']

###### Funcion para imputar con la moda

In [ ]:
def imputar_moda(variable):
    return(variable.fillna(variable.mode()[0]))

###### Imputar con la moda

In [ ]:
cat[var_imputar_moda] = cat[var_imputar_moda].apply(imputar_moda)

### Atipicos

#### Categorias poco frecuentes

In [ ]:
cat.info()

In [ ]:
cat.ambito.value_counts()

##### Función para agrupar categorías raras en 'OTROS'

In [ ]:
def agrupar_cat_raras(variable, criterio = 0.02):
    #Calcula las frecuencias
    frecuencias = variable.value_counts(normalize=True)
    #Identifica las que estan por debajo del criterio
    temp = [cada for cada in frecuencias.loc[frecuencias < criterio].index.values]
    #Las recodifica en 'OTROS'
    temp2 = np.where(variable.isin(temp),'OTROS',variable)
    #Devuelve el resultado
    return(temp2)

##### Variables a imputar

In [ ]:
cat.columns.to_list()

In [ ]:
var_imputar_cat_raras =cat.columns.to_list()

##### Aplicar

In [ ]:
for variable in var_imputar_cat_raras:
    cat[variable] = agrupar_cat_raras(cat[variable],criterio = 0.02)

## GESTION DE NUMERICAS

### Nulos

#### Identificacion

In [ ]:
num.isna().sum().sort_values(ascending = False)

#### Correccion

##### Imputar por la mediana

###### Variables a imputar

In [ ]:
var_imputar_mediana = num.columns.to_list()

###### Funcion para imputar con la mediana

In [ ]:
def imputar_mediana(variable):
    if pd.api.types.is_integer_dtype(variable):
        return(variable.fillna(int(variable.median())))
    else:
        return(variable.fillna(variable.median()))

###### Imputar con la mediana

In [ ]:
num[var_imputar_mediana] = num[var_imputar_mediana].apply(imputar_mediana)

### Atipicos

#### Por desviacion tipica

##### Identificacion

###### Definir el numero de desviaciones tipicas

In [ ]:
num_desv_tip = 4

###### Funcion que devuelve los indices de los atipicos

In [ ]:
def atipicos_desv_tip(variable, num_desv_tip = 4):
    #sacamos los nulos por ahora
    variable = variable.dropna()
    #calculamos los límites
    media = np.mean(variable)
    sd = np.std(variable)
    umbral = sd * num_desv_tip
    lim_inf = media - umbral
    lim_sup = media + umbral
    #encontramos los índices de los que estan fuera de los limites
    indices = [indice for indice,valor in variable.items() if valor < lim_inf or valor > lim_sup]
    return(indices)

###### Funcion que cuenta el numero de atipicos

In [ ]:
def conteo_atipicos(df,variable, num_desv_tip = 4):
    atipicos = atipicos_desv_tip(df[variable], num_desv_tip)
    return(df.loc[atipicos,variable].value_counts())

###### Variables a aplicar

In [ ]:
var_atipicos_dt = num.columns.to_list()

###### Aplicar

In [ ]:
for variable in var_atipicos_dt:
    print('\n' + variable + ':\n')
    print(conteo_atipicos(num,variable,num_desv_tip))

#### Por winsorizacion manual para una variable

In [ ]:
num

In [ ]:
num['visitas_total'] = num['visitas_total'].clip(0,50)
num['paginas_vistas_visita'] = num['paginas_vistas_visita'].clip(0,20)
#num.clip({'visitas_total': (0,50)}, inplace=True)
#num.clip({'paginas_vistas_visita': (0,20)}, inplace=True)

In [ ]:
num.visitas_total.value_counts()

##### Identificacion

## GUARDAR DATASETS TRAS CALIDAD DE DATOS

Guarda los avances en df, cat y num con un sufijo para poder guardar o recuperar avances del proyecto.

En formato pickle para no perder las modificaciones de metadatos.

In [ ]:
num

### Definir los nombres de los archivos

In [ ]:
ruta_trabajo = ruta_proyecto + '/02_Datos/03_Trabajo/' + 'trabajo_resultado_calidad.pickle'
ruta_cat = ruta_proyecto + '/02_Datos/03_Trabajo/' + 'cat_resultado_calidad.pickle'
ruta_num = ruta_proyecto + '/02_Datos/03_Trabajo/' + 'num_resultado_calidad.pickle'

### Guardar los archivos

In [ ]:
df.to_pickle(ruta_trabajo)
cat.to_pickle(ruta_cat)
num.to_pickle(ruta_num)